## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-20-13-55-06 +0000,nyt,Britain Announces Brexit Reset Deal at E.U. Su...,https://www.nytimes.com/2026/11/20/world/europ...
1,2026-11-20-13-52-27 +0000,nypost,Trump to talk Iran war in keynote address at R...,https://nypost.com/2026/11/20/us-news/trump-to...
2,2026-11-20-13-52-04 +0000,bbc,Russian missile hits Poland as Ukraine's allie...,https://www.bbc.co.uk/news/articles/cqx2x9dklx...
3,2026-11-20-13-48-48 +0000,nyt,Anthropic Shares Slip as AI and Tech Stock Sel...,https://www.nytimes.com/2026/11/20/business/an...
4,2026-11-20-13-48-16 +0000,wapo,White House replaces top cybersecurity officia...,https://www.washingtonpost.com/politics/2026/1...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2534/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
6,trump,47
26,anthropic,39
115,new,20
257,putin,18
29,ai,18


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
326,2026-11-19-19-03-56 +0000,nypost,China sent its tallest official to tower over ...,https://nypost.com/2026/11/19/world-news/china...,126
338,2026-11-19-18-02-00 +0000,wsj,"Trump, Xi, Putin End Historic Summit With Denu...",https://www.wsj.com/world/trump-xi-and-putin-c...,119
209,2026-11-19-23-23-22 +0000,wapo,"In summit with Trump and Putin, Xi put himself...",https://www.washingtonpost.com/world/2026/11/1...,111
256,2026-11-19-21-44-18 +0000,wapo,Trump says Putin wants peace in Ukraine even a...,https://www.washingtonpost.com/world/2026/11/1...,107
361,2026-11-19-17-23-29 +0000,bbc,"BBC correspondents discuss what Trump, Xi and ...",https://www.bbc.co.uk/news/videos/c63rgnq05pdw...,106


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
326,126,2026-11-19-19-03-56 +0000,nypost,China sent its tallest official to tower over ...,https://nypost.com/2026/11/19/world-news/china...
195,104,2026-11-20-00-00-00 +0000,wsj,Anthropic’s Mega IPO Lives Up to Wall Street H...,https://www.wsj.com/finance/stocks/anthropic-i...
151,93,2026-11-20-02-54-11 +0000,nypost,White House taps Kirsten Davies as new cyber c...,https://nypost.com/2026/11/19/us-news/wh-names...
296,80,2026-11-19-20-07-02 +0000,bbc,Myanmar committed genocide against Rohingya mi...,https://www.bbc.co.uk/news/articles/c804wzp20d...
245,68,2026-11-19-22-06-00 +0000,wsj,Senate Republicans are suspending their effort...,https://www.wsj.com/politics/policy/pentagon-m...
203,53,2026-11-19-23-41-09 +0000,nypost,California AG Rob Bonta serves Nvidia with sub...,https://nypost.com/2026/11/19/business/califor...
140,48,2026-11-20-04-12-00 +0000,wsj,The Senate passed a bipartisan bill designed t...,https://www.wsj.com/politics/policy/senate-pas...
40,45,2026-11-20-12-00-00 +0000,wsj,Vice President JD Vance has whipsawed between ...,https://www.wsj.com/politics/trump-says-vance-...
236,44,2026-11-19-22-34-11 +0000,chitri,"Will County Board OKs no levy increase, reject...",https://www.chicagotribune.com/2026/11/19/will...
6,44,2026-11-20-13-38-21 +0000,nypost,Jared Kushner says Russia appears ready to acc...,https://nypost.com/2026/11/20/us-news/jared-ku...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
